# Prepare gene feature for matching analysis
For overlap with genes, we download the Gencode gene set release 50 (https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/gencode.v50.basic.annotation.gtf.gz)

In [1]:
import pandas as pd

In [2]:
# columns derived from https://www.gencodegenes.org/pages/data_format.html
gff_df = pd.read_csv(
    "supporting_data/gencode.v50.basic.annotation.gtf.gz",
    compression='gzip',
    delimiter='\t',
    header=4,
    names=[
        'chromosome', 'annotation_source', 'feature_type',
        'genomic_start_location', 'genomic_end_location',
        'score', 'genomic_strand', 'genomic_phase', 'additional_info'
    ]
)

In [3]:
gff_df = (
    gff_df
    .query("feature_type == 'gene'")
    .reset_index()
)

In [4]:
# extract additional_info fields; stored as a single string but contains key/value pairs

gff_df['info'] = (
    gff_df['additional_info']
    .str.split('; ') # split main string into key/value attribute strings
    .apply(lambda items: [item.split(' ') for item in items]) # split into key/value tuples
    .apply(dict)
)
info_df = pd.json_normalize(gff_df['info'])
for col in info_df.columns:
    info_df[col] = info_df[col].str.strip('\"') # remove quotation mark characters wrapped around values

In [5]:
gene_df = pd.concat(
    [
        gff_df,
        info_df
    ],
    axis=1
)

In [6]:
gene_df = gene_df.query("gene_type == 'protein_coding'")

In [7]:
gene_df['genomic_start_location'] = gene_df['genomic_start_location'].astype(int)
gene_df['genomic_end_location'] = gene_df['genomic_end_location'].astype(int)

In [8]:
gene_df[['chromosome', 'genomic_start_location', 'genomic_end_location', 'gene_id', 'gene_name']]

,chromosome,genomic_start_location,genomic_end_location,gene_id,gene_name
12,chr1,65419,71585,ENSG00000186092.7,OR4F5
39,chr1,450740,451678,ENSG00000284733.2,OR4F29
56,chr1,685716,686654,ENSG00000284662.2,OR4F16
76,chr1,923895,944590,ENSG00000187634.15,SAMD11
77,chr1,943527,960714,ENSG00000188976.13,NOC2L
...,...,...,...,...,...
78722,chrM,10470,10766,ENSG00000212907.2,MT-ND4L
78723,chrM,10760,12137,ENSG00000198886.2,MT-ND4
78727,chrM,12337,14148,ENSG00000198786.2,MT-ND5
78728,chrM,14149,14673,ENSG00000198695.2,MT-ND6


In [9]:
gene_df.to_csv(
    'supporting_data/gene-feature-locations-preprocessed.csv',
    index=False
)